# 🤖 Z32LITE Fine-Tuning Notebook
**Model:** Qwen/Qwen2.5-1.5B-Instruct  
**Technique:** QLoRA (4-bit quantization)  
**License:** Apache 2.0 (Commercial + Open Use)  
**Target:** Low-end Android devices (~2-3GB RAM)  

---
## Steps
1. Install dependencies
2. Load model in 4-bit (QLoRA)
3. Load and prepare dataset
4. Train with LoRA adapters
5. Save and export to GGUF format

> ⚠️ Make sure you selected **GPU (T4)** runtime in Colab!

In [ ]:
# =============================================
# CELL 1: Install Dependencies
# =============================================
!pip install -q transformers==4.44.0 datasets trl peft bitsandbytes accelerate
!pip install -q huggingface_hub sentencepiece
print('✅ Dependencies installed!')

In [ ]:
# =============================================
# CELL 2: GPU Check
# =============================================
import torch

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {device_name} ({total_mem:.1f} GB)')
else:
    print('❌ No GPU! Please switch to GPU runtime.')

In [ ]:
# =============================================
# CELL 3: Login to HuggingFace (to push model)
# =============================================
# Optional: Only needed if you want to push to your HuggingFace Hub
# from huggingface_hub import notebook_login
# notebook_login()
print('(Skip this step if you only want to download the GGUF file locally)')

In [ ]:
# =============================================
# CELL 4: Config - Edit these values!
# =============================================

MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"   # Base model (Apache 2.0)
RUN_NAME   = "z32lite-alpha-1"               # Name for this training run
OUTPUT_DIR = "./z32lite_output"              # Where to save checkpoints

# LoRA Hyperparameters
LORA_R     = 64    # Rank - higher = smarter but heavier
LORA_ALPHA = 16    # Alpha - scaling factor
LORA_DROPOUT = 0.05

# Training Hyperparameters
MAX_SEQ_LENGTH = 2048
BATCH_SIZE     = 2      # Keep low for T4 (15GB)
GRAD_ACCUM     = 4      # Effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3
WARMUP_RATIO   = 0.03

print(f'✅ Config set: {MODEL_ID}')
print(f'   LoRA rank: {LORA_R}, LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}')

In [ ]:
# =============================================
# CELL 5: Load Model & Tokenizer in 4-bit
# =============================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print('✅ Model loaded in 4-bit!')

# LoRA Config
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA adapters attached!')

In [ ]:
# =============================================
# CELL 6: Define Dataset Format
# =============================================
# We use the Qwen chat format: <|im_start|>role\ncontent<|im_end|>
# Each example has a 'conversations' field with a list of {from, value} dicts

# --- EXAMPLE DATA --- Replace with your real dataset!
raw_data = [
    # ===== GENERAL ARABIC CONVERSATIONS =====
    {
        "conversations": [
            {"from": "user",  "value": "إيه أحسن طريقة أنام بسرعة؟"},
            {"from": "assistant", "value": "أهم حاجة تضبط مواعيد نومك، تبعد عن الشاشات قبل النوم بساعة، وتعمل حمام دافي أو تسمع موسيقى هادية. الغرفة لازم تكون مظلمة وباردة شوية."}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "ما هي أسرع طريقة لتعلم البرمجة؟"},
            {"from": "assistant", "value": "أسرع طريقة هي إنك تختار لغة واحدة بس في الأول، وتبني بيها مشاريع حقيقية. مثلاً Python مع مشروع بسيط زي برنامج يحسب المصروفات. التطبيق العملي أسرع من مجرد مشاهدة الفيديوهات."}
        ]
    },
    # ===== ANDROID CONTROL (Function Calling) =====
    {
        "conversations": [
            {"from": "user",  "value": "عالي الصوت"},
            {"from": "assistant", "value": "SYSTEM_ACTION:{\"action\":\"set_volume\",\"direction\":\"up\"}"}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "وطي صوت الموسيقى"},
            {"from": "assistant", "value": "SYSTEM_ACTION:{\"action\":\"set_volume\",\"direction\":\"down\",\"stream\":\"media\"}"}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "غير الأغنية"},
            {"from": "assistant", "value": "SYSTEM_ACTION:{\"action\":\"media_next_track\"}"}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "دور على رقم دكتور كمال في جهات الاتصال"},
            {"from": "assistant", "value": "SYSTEM_ACTION:{\"action\":\"search_contacts\",\"query\":\"دكتور كمال\"}"}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "فين أقرب صيدلية دلوقتي؟"},
            {"from": "assistant", "value": "SYSTEM_ACTION:{\"action\":\"search_web\",\"query\":\"أقرب صيدلية\"}"}
        ]
    },
    # ===== NOTIFICATION HANDLING =====
    {
        "conversations": [
            {"from": "user",  "value": "لو جالي ايميل من الشغل خبرني"},
            {"from": "assistant", "value": "تمام! هاشوف الإشعارات وأول ما يجيلك ايميل على الشغل، هبعتلك تنبيه فوراً."}
        ]
    },
    # ===== MULTILINGUAL =====
    {
        "conversations": [
            {"from": "user",  "value": "What is the capital of Egypt?"},
            {"from": "assistant", "value": "The capital of Egypt is Cairo (القاهرة). It is the largest city in Africa and the Arab world, with a population of over 20 million people."}
        ]
    },
    {
        "conversations": [
            {"from": "user",  "value": "ترجملي الجملة دي: The weather is nice today"},
            {"from": "assistant", "value": "الترجمة: \"الجو حلو النهاردة\" 🌤️"}
        ]
    },
]

print(f'✅ Dataset loaded: {len(raw_data)} examples')
print('⚠️  Remember to replace raw_data with your full dataset for better results!')

In [ ]:
# =============================================
# CELL 7: Format Dataset to Qwen Chat Template
# =============================================
from datasets import Dataset

def format_conversation(example):
    """Convert conversation to Qwen chat format string."""
    messages = []
    # Add system prompt
    messages.append({
        "role": "system",
        "content": (
            "أنت Z32LITE، مساعد ذكاء اصطناعي خفيف وسريع مصمم للأجهزة الاقتصادية. "
            "تتحدث العربية والإنجليزية وتفهم اللهجة المصرية. "
            "عندما تحتاج لتنفيذ إجراء على الجهاز، استخدم الصيغة: SYSTEM_ACTION:{json_object}. "
            "عندما تحتاج للبحث في الإنترنت، استخدم: SYSTEM_ACTION:{\"action\":\"search_web\",\"query\":\"...\"}."
        )
    })
    for turn in example['conversations']:
        role = "user" if turn['from'] == 'user' else "assistant"
        messages.append({"role": role, "content": turn['value']})
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_conversation, remove_columns=['conversations'])

print('Sample formatted example:')
print('-' * 50)
print(dataset[0]['text'])
print('-' * 50)
print(f'✅ Dataset formatted: {len(dataset)} examples')

In [ ]:
# =============================================
# CELL 8: Train!
# =============================================
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

In [ ]:
# =============================================
# CELL 9: Save Fine-tuned Adapters
# =============================================
SAVE_PATH = "./z32lite_lora_adapters"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ LoRA adapters saved to: {SAVE_PATH}')

In [ ]:
# =============================================
# CELL 10: Merge Model + Export to GGUF
# (Run only after training is done)
# =============================================
# Step 1: Merge LoRA adapters into the base model
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('Loading base model for merging...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cpu"  # Use CPU for merging to avoid OOM
)

print('Merging LoRA adapters...')
merged_model = PeftModel.from_pretrained(base_model, SAVE_PATH)
merged_model = merged_model.merge_and_unload()

MERGED_PATH = "./z32lite_merged"
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True, max_shard_size="4GB")
tokenizer.save_pretrained(MERGED_PATH)
print(f'✅ Merged model saved to: {MERGED_PATH}')

In [ ]:
# =============================================
# CELL 11: Convert to GGUF (4-bit Q4_K_M)
# This is what will run on Android!
# =============================================
# Install llama.cpp for conversion
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt

# Convert to GGUF format
!python llama.cpp/convert_hf_to_gguf.py ./z32lite_merged --outtype f16 --outfile z32lite_f16.gguf

# Quantize to 4-bit (Q4_K_M) - this is the sweet spot between quality and size
!cd llama.cpp && make quantize
!./llama.cpp/quantize z32lite_f16.gguf z32lite_Q4_K_M.gguf Q4_K_M

print('✅ GGUF files created!')
!ls -lh z32lite*.gguf

In [ ]:
# =============================================
# CELL 12: Test the model before export
# =============================================
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto"
)

def chat(user_input):
    messages = [
        {"role": "system", "content": "أنت Z32LITE مساعد ذكاء اصطناعي سريع وذكي."},
        {"role": "user",   "content": user_input}
    ]
    output = pipe(messages, max_new_tokens=256, do_sample=True, temperature=0.7)
    return output[0]['generated_text'][-1]['content']

# Test cases
tests = [
    "مرحباً! إيه اسمك؟",
    "عالي الصوت",
    "فين أقرب صيدلية؟",
]

for test in tests:
    print(f'\n👤 User: {test}')
    print(f'🤖 Z32LITE: {chat(test)}')
    print('-' * 40)

## ✅ Next Steps After Export

1. **Download** `z32lite_Q4_K_M.gguf` from Colab (Files panel on the left)
2. **Integrate** into Android app using `llama.cpp` Android bindings (JNI)
3. **Test** inference speed on target device

Expected size: **~900MB - 1.1GB** for Q4_K_M
Expected RAM usage: **~1.2GB** on device
Expected speed: **5-15 tokens/sec** on mid-range device